# Experiment Hub — Stock Fraud Screener

Personal research dashboard. Covers the full analytical stack in one place:

| Section | What it shows |
|---------|---------------|
| **1. Feature Selection** | IC / ICIR rankings, SHAP, selected features per horizon |
| **2. Model Performance** | OOF AUC, walk-forward AUC, regression IC, val/test split |
| **3. Screener Rankings** | All strategies ranked by Sharpe, CAGR, IC |
| **4. Screener Deep Dive** | Full profile for one screener (pick in Config) |
| **5. Live Picks** | Quality-gated long/short candidates for today |

---
**Workflow:** set Config → Run All → inspect each section.

## 0. Config — set everything here

In [ ]:
from pathlib import Path

ROOT = Path("..")  # repo root relative to notebooks/

# ── Section 1 & 2: Feature / Model analysis ───────────────────────────────────
# Horizons to compare. Options: '6m' | '1y' | '2y' | '3y' | '5y'
HORIZONS = ['6m', '1y', '2y', '3y', '5y']

# How many top features to show in charts
TOP_FEATURES_N = 20

# ── Section 3: Screener rankings ─────────────────────────────────────────────
# Rank screeners by this metric. Options: 'sharpe' | 'cagr_pct' | 'ic_mean' | 'icir'
RANK_BY = 'sharpe'

# ── Section 4: Single screener deep dive ─────────────────────────────────────
# Pick one signal_id from alpha_registry.json.
# Options: 'alpha_value' | 'alpha_quality' | 'alpha_momentum' | 'alpha_growth'
#          'alpha_fraud_risk' | 'ml_1y_oof' | 'ml_3y_oof' | 'ml_5y_oof'
SCREENER_ID = 'ml_3y_oof'

# ── Section 5: Live picks ─────────────────────────────────────────────────────
USE_HF          = False           # True = pull from HuggingFace instead of local
HF_REPO         = 'ekrash718/stock-fraud-screener'
MARKETS         = None            # None = all; or ['US', 'KR', 'JP', 'BR', 'CA']
LIVE_HORIZON    = '3y'
TOP_N_LONG      = 25
TOP_N_SHORT     = 15
MIN_MARKET_CAP  = 10_000_000
MAX_MARKET_CAP  = 300_000_000
PIOTROSKI_MIN   = 5
BENEISH_MAX     = -1.78
ALTMAN_MIN      = 1.81
PB_MAX          = 3.0
FCF_YIELD_MIN   = 0.03
APPLY_MOS       = True
BETA_MAX        = 0.80
HALF_KELLY      = 0.25
MAX_LEVERAGE    = 2.0
POSITION_CAP    = 0.05
SECTOR_CAP      = 0.40

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.colors as mcolors

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 60)

# Paths
REPORTS = ROOT / 'reports'
MODELS  = ROOT / 'models'
DATA    = ROOT / 'data'

print('Paths OK:')
for p in [REPORTS, MODELS, DATA]:
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'  {p}  {status}')

---
## 1. Feature Selection Dashboard

In [ ]:
# ── Load feature selection summary ───────────────────────────────────────────
fs_path = REPORTS / 'feature_selection_summary.csv'
fs = pd.read_csv(fs_path)
print(f"Feature selection summary: {fs.shape}  horizons: {fs['horizon'].unique()}")
fs.head(3)

In [ ]:
# ── Selected features per horizon (from feature_sets JSON) ───────────────────
selected_by_horizon = {}
for h in HORIZONS:
    p = MODELS / f'feature_sets_{h}.json'
    if p.exists():
        selected_by_horizon[h] = json.loads(p.read_text())
    else:
        selected_by_horizon[h] = []

print('Selected feature counts per horizon:')
for h, feats in selected_by_horizon.items():
    print(f'  {h}: {len(feats)} features')

In [ ]:
# ── IC / ICIR table for the chosen HORIZONS ───────────────────────────────────
cols_to_show = ['feature', 'horizon', 'mean_ic', 'std_ic', 'icir', 'ic_tstat_nw',
                'fdr_reject', 'psi_train_vs_test', 'selected']
present = [c for c in cols_to_show if c in fs.columns]

ic_table = (
    fs[fs['horizon'].isin(HORIZONS)][present]
    .sort_values(['horizon', 'icir'], ascending=[True, False])
)

print(f'Features per horizon (IC > 0):')
for h in HORIZONS:
    sub = ic_table[ic_table['horizon'] == h]
    sel = sub['selected'].sum() if 'selected' in sub.columns else 'N/A'
    pos = (sub['mean_ic'] > 0).sum() if 'mean_ic' in sub.columns else 'N/A'
    print(f'  {h}: {len(sub)} total  |  {pos} positive IC  |  {sel} selected')

ic_table[ic_table['horizon'] == HORIZONS[-2]].head(20)  # show one horizon

In [ ]:
# ── ICIR bar chart: top N features for each horizon ──────────────────────────
n_horizons = len(HORIZONS)
fig, axes = plt.subplots(1, n_horizons, figsize=(5 * n_horizons, 6), sharey=False)
if n_horizons == 1:
    axes = [axes]

for ax, h in zip(axes, HORIZONS):
    sub = (
        fs[(fs['horizon'] == h) & fs['mean_ic'].notna()]
        .nlargest(TOP_FEATURES_N, 'icir')
        .sort_values('icir')
    )
    colors = ['#2ecc71' if s else '#bdc3c7' for s in sub.get('selected', [False]*len(sub))]
    ax.barh(sub['feature'], sub['icir'], color=colors, edgecolor='none')
    ax.axvline(0.5, color='orange', linestyle='--', linewidth=1, label='ICIR=0.5 threshold')
    ax.set_title(f'ICIR — horizon={h}\n(green = selected)', fontsize=10)
    ax.set_xlabel('ICIR')
    ax.tick_params(axis='y', labelsize=7)

plt.suptitle(f'Top {TOP_FEATURES_N} Features by ICIR', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP importance per horizon ───────────────────────────────────────────────
shap_frames = {}
for h in HORIZONS:
    p = REPORTS / f'shap_importance_{h}.csv'
    if p.exists():
        shap_frames[h] = pd.read_csv(p).head(TOP_FEATURES_N)

fig, axes = plt.subplots(1, len(shap_frames), figsize=(5 * len(shap_frames), 6), sharey=False)
if len(shap_frames) == 1:
    axes = [axes]

for ax, (h, df_s) in zip(axes, shap_frames.items()):
    df_s = df_s.sort_values('shap_mean_abs')
    ax.barh(df_s['feature'], df_s['shap_mean_abs'], color='steelblue', edgecolor='none')
    ax.set_title(f'SHAP — horizon={h}', fontsize=10)
    ax.set_xlabel('Mean |SHAP|')
    ax.tick_params(axis='y', labelsize=7)

plt.suptitle(f'Top {TOP_FEATURES_N} Features by Mean |SHAP|', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature overlap heatmap across horizons (selected features) ───────────────
all_selected = sorted(set(f for feats in selected_by_horizon.values() for f in feats))
overlap = pd.DataFrame(
    {h: [1 if f in selected_by_horizon[h] else 0 for f in all_selected]
     for h in HORIZONS},
    index=all_selected
)

if len(all_selected) > 0:
    # Sort by total appearances
    overlap['_total'] = overlap.sum(axis=1)
    overlap = overlap.sort_values('_total', ascending=False).drop(columns='_total')

    fig, ax = plt.subplots(figsize=(len(HORIZONS) * 1.2 + 2, max(6, len(overlap) * 0.35)))
    im = ax.imshow(overlap.values, cmap='Greens', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(HORIZONS)))
    ax.set_xticklabels(HORIZONS, fontsize=10)
    ax.set_yticks(range(len(all_selected)))
    ax.set_yticklabels(overlap.index, fontsize=7)
    ax.set_title('Feature Selection Overlap Across Horizons\n(green = selected)', fontsize=11)
    plt.colorbar(im, ax=ax, shrink=0.5)
    plt.tight_layout()
    plt.show()
    print(f'\nFeatures selected in ALL horizons: {(overlap.sum(axis=1) == len(HORIZONS)).sum()}')
    print(f'Features selected in exactly 1 horizon: {(overlap.sum(axis=1) == 1).sum()}')
else:
    print('No feature_sets JSON found.')

---
## 2. Model Performance Dashboard

In [ ]:
# ── Load model_meta.json ─────────────────────────────────────────────────────
meta = json.loads((MODELS / 'model_meta.json').read_text())

perf_rows = []
for h in HORIZONS:
    m = meta.get(h, {})
    perf_rows.append({
        'horizon': h,
        'n_train': m.get('n_train'),
        'val_auc': m.get('val_auc'),
        'test_auc': m.get('test_auc'),
        'lr_val_auc': m.get('lr_val_auc'),
        'lr_test_auc': m.get('lr_test_auc'),
        'wf_mean_auc': m.get('wf_mean_auc'),
        'n_features': len(m.get('features', [])),
        'pos_rate': m.get('pos_rate'),
    })

perf_df = pd.DataFrame(perf_rows).set_index('horizon')
print('Model performance summary from model_meta.json:')
perf_df

In [ ]:
# ── OOF AUC folds per horizon ─────────────────────────────────────────────────
oof_frames = {}
for h in HORIZONS:
    p = REPORTS / f'oof_auc_{h}.csv'
    if p.exists():
        df_o = pd.read_csv(p)
        df_o['horizon'] = h
        oof_frames[h] = df_o

oof_all = pd.concat(oof_frames.values(), ignore_index=True) if oof_frames else pd.DataFrame()

if not oof_all.empty:
    print('OOF AUC by horizon (mean ± std):')
    print(oof_all.groupby('horizon')['auc'].agg(['mean', 'std', 'min', 'max']).round(4))

In [ ]:
# ── Walk-forward AUC per horizon ──────────────────────────────────────────────
wf_frames = {}
for h in HORIZONS:
    p = REPORTS / f'walk_forward_auc_{h}.csv'
    if p.exists():
        df_w = pd.read_csv(p)
        df_w['horizon'] = h
        wf_frames[h] = df_w

wf_all = pd.concat(wf_frames.values(), ignore_index=True) if wf_frames else pd.DataFrame()

if not wf_all.empty:
    print('Walk-forward AUC by horizon (mean ± std):')
    print(wf_all.groupby('horizon')['auc'].agg(['mean', 'std', 'min', 'max']).round(4))

In [ ]:
# ── Regression IC per horizon ────────────────────────────────────────────────
reg_frames = {}
for h in HORIZONS:
    p = REPORTS / f'regression_ic_{h}.csv'
    if p.exists():
        df_r = pd.read_csv(p)
        df_r['horizon'] = h
        reg_frames[h] = df_r

reg_all = pd.concat(reg_frames.values(), ignore_index=True) if reg_frames else pd.DataFrame()

if not reg_all.empty:
    ic_col = [c for c in ['spearman_ic', 'ic', 'pearson_ic'] if c in reg_all.columns]
    if ic_col:
        ic_col = ic_col[0]
        print(f'Regression IC ({ic_col}) by horizon (mean ± std):')
        print(reg_all.groupby('horizon')[ic_col].agg(['mean', 'std', 'min', 'max']).round(4))

In [ ]:
# ── Combined model performance chart ─────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. AUC comparison: val vs test vs walk-forward
ax = axes[0, 0]
x = range(len(HORIZONS))
ax.bar([i - 0.25 for i in x], perf_df['val_auc'],  width=0.22, label='Val AUC',  color='steelblue', alpha=0.85)
ax.bar([i + 0.00 for i in x], perf_df['test_auc'], width=0.22, label='Test AUC', color='coral',     alpha=0.85)
ax.bar([i + 0.25 for i in x], perf_df['wf_mean_auc'], width=0.22, label='WF Mean AUC', color='seagreen', alpha=0.85)
ax.axhline(0.55, color='orange', linestyle='--', linewidth=1, label='AUC=0.55')
ax.axhline(0.50, color='gray',   linestyle=':',  linewidth=1)
ax.set_xticks(list(x))
ax.set_xticklabels(HORIZONS)
ax.set_ylabel('AUC')
ax.set_title('Val / Test / Walk-Forward AUC')
ax.legend(fontsize=8)
ax.set_ylim(0.45, 0.70)

# 2. OOF AUC over time (walk-forward folds)
ax = axes[0, 1]
if not oof_all.empty and 'test_year' in oof_all.columns:
    for h in HORIZONS:
        sub = oof_all[oof_all['horizon'] == h].sort_values('test_year')
        ax.plot(sub['test_year'], sub['auc'], marker='o', markersize=4, label=h)
    ax.axhline(0.50, color='gray', linestyle=':', linewidth=1)
    ax.set_xlabel('Test Year')
    ax.set_ylabel('OOF AUC')
    ax.set_title('OOF AUC Over Time by Horizon')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'OOF AUC data unavailable', ha='center', va='center', transform=ax.transAxes)

# 3. Walk-forward AUC over time
ax = axes[1, 0]
if not wf_all.empty and 'test_year' in wf_all.columns:
    for h in HORIZONS:
        sub = wf_all[wf_all['horizon'] == h].sort_values('test_year')
        ax.plot(sub['test_year'], sub['auc'], marker='s', markersize=4, label=h)
    ax.axhline(0.50, color='gray', linestyle=':', linewidth=1)
    ax.set_xlabel('Test Year')
    ax.set_ylabel('Walk-Forward AUC')
    ax.set_title('Walk-Forward AUC Over Time')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'Walk-forward AUC data unavailable', ha='center', va='center', transform=ax.transAxes)

# 4. Regression IC over time
ax = axes[1, 1]
if not reg_all.empty and 'test_year' in reg_all.columns:
    ic_col_name = [c for c in ['spearman_ic', 'ic', 'pearson_ic'] if c in reg_all.columns]
    if ic_col_name:
        ic_col_name = ic_col_name[0]
        for h in HORIZONS:
            sub = reg_all[reg_all['horizon'] == h].sort_values('test_year')
            ax.plot(sub['test_year'], sub[ic_col_name], marker='^', markersize=4, label=h)
        ax.axhline(0.0, color='gray', linestyle=':', linewidth=1)
        ax.set_xlabel('Test Year')
        ax.set_ylabel('Spearman IC')
        ax.set_title('Regression Model IC Over Time')
        ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'Regression IC data unavailable', ha='center', va='center', transform=ax.transAxes)

plt.suptitle('Model Performance Dashboard', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Screener Rankings — All Strategies

In [ ]:
# ── Load alpha registry ───────────────────────────────────────────────────────
registry = json.loads((DATA / 'alpha_registry.json').read_text())
signals = pd.DataFrame(registry['signals'])
print(f"Registry generated: {registry.get('generated_at','?')}")
print(f"Total signals: {len(signals)}  |  Selected: {signals['selected'].sum()}")
signals.head(3)

In [ ]:
# ── Full rankings table ───────────────────────────────────────────────────────
rank_cols = [
    'signal_id', 'category', 'selected',
    'ic_mean', 'icir',
    'cagr_pct', 'cagr_bootstrap_mean_pct', 'cagr_bootstrap_1sigma_pct',
    'sharpe', 'sharpe_bootstrap_mean',
    'sortino', 'calmar',
    'max_drawdown_pct', 'excess_cagr_vs_spy',
    'beta_vs_spy', 'hit_rate_pct', 'n_years', 'top_n', 'cost_bps'
]
present_rank = [c for c in rank_cols if c in signals.columns]
rankings = signals[present_rank].sort_values(RANK_BY, ascending=False).reset_index(drop=True)
rankings.index = range(1, len(rankings) + 1)

print(f'All screeners ranked by {RANK_BY} (descending):')
rankings

In [ ]:
# ── Screener rankings chart ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

palette = ['#2ecc71' if s else '#e74c3c' for s in signals['selected']]

# CAGR bar
ax = axes[0]
sorted_s = signals.sort_values('cagr_pct', ascending=True)
colors = ['#2ecc71' if s else '#e74c3c' for s in sorted_s['selected']]
ax.barh(sorted_s['signal_id'], sorted_s['cagr_pct'], color=colors, edgecolor='none')
ax.axvline(0, color='gray', linewidth=0.8)
ax.set_xlabel('CAGR (%)')
ax.set_title('CAGR by Screener\n(green = in composite)')

# Sharpe bar
ax = axes[1]
sorted_s2 = signals.sort_values('sharpe', ascending=True)
colors2 = ['#2ecc71' if s else '#e74c3c' for s in sorted_s2['selected']]
ax.barh(sorted_s2['signal_id'], sorted_s2['sharpe'], color=colors2, edgecolor='none')
ax.axvline(0.5, color='orange', linestyle='--', linewidth=1, label='Sharpe=0.5')
ax.axvline(0, color='gray', linewidth=0.8)
ax.set_xlabel('Sharpe Ratio')
ax.set_title('Sharpe by Screener')
ax.legend(fontsize=8)

# CAGR vs Sharpe scatter
ax = axes[2]
for _, row in signals.iterrows():
    c = '#2ecc71' if row.get('selected') else '#e74c3c'
    ax.scatter(row.get('sharpe', np.nan), row.get('cagr_pct', np.nan),
               color=c, s=100, zorder=3)
    ax.annotate(row['signal_id'], (row.get('sharpe', 0), row.get('cagr_pct', 0)),
                fontsize=7, xytext=(4, 2), textcoords='offset points')
ax.axvline(0.5, color='orange', linestyle='--', linewidth=1)
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xlabel('Sharpe')
ax.set_ylabel('CAGR (%)')
ax.set_title('CAGR vs Sharpe\n(green = in composite)')

plt.suptitle('Screener Rankings Overview', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Risk-adjusted metrics summary ─────────────────────────────────────────────
risk_cols = ['signal_id', 'sharpe', 'sortino', 'calmar', 'max_drawdown_pct',
             'excess_cagr_vs_spy', 'beta_vs_spy', 'hit_rate_pct']
risk_present = [c for c in risk_cols if c in signals.columns]
risk_table = signals[risk_present].sort_values('sharpe', ascending=False).reset_index(drop=True)
risk_table.index = range(1, len(risk_table) + 1)
print('Risk-adjusted metrics (sorted by Sharpe):')
risk_table

---
## 4. Single Screener Deep Dive
> **Set `SCREENER_ID` in Config cell to change the target screener.**

In [ ]:
# ── Load screener profile from registry ──────────────────────────────────────
screener_matches = signals[signals['signal_id'] == SCREENER_ID]

if screener_matches.empty:
    print(f"⚠ SCREENER_ID='{SCREENER_ID}' not found in alpha_registry.")
    print(f"Available: {signals['signal_id'].tolist()}")
else:
    scr = screener_matches.iloc[0]
    print(f"{'═'*55}")
    print(f"  SCREENER: {scr['signal_id']}")
    print(f"{'═'*55}")
    show_fields = [
        'category', 'horizon', 'market', 'selected',
        'ic_mean', 'icir',
        'cagr_pct', 'cagr_bootstrap_mean_pct', 'cagr_bootstrap_1sigma_pct',
        'sharpe', 'sharpe_bootstrap_mean', 'sharpe_bootstrap_1sigma',
        'sortino', 'calmar',
        'max_drawdown_pct', 'excess_cagr_vs_spy', 'beta_vs_spy',
        'hit_rate_pct', 'n_years', 'top_n', 'cost_bps',
    ]
    for f in show_fields:
        if f in scr and pd.notna(scr[f]):
            print(f"  {f:35} {scr[f]}")

    if 'features_used' in scr and isinstance(scr['features_used'], list):
        print(f"\n  Features used ({len(scr['features_used'])}):\n  {scr['features_used']}")

In [ ]:
# ── Screener vs peers — ranked comparison chart ───────────────────────────────
if not screener_matches.empty:
    metrics = ['sharpe', 'cagr_pct', 'sortino', 'calmar', 'excess_cagr_vs_spy', 'icir']
    metrics_avail = [m for m in metrics if m in signals.columns]

    fig, axes = plt.subplots(1, len(metrics_avail), figsize=(3 * len(metrics_avail), 5))
    if len(metrics_avail) == 1:
        axes = [axes]

    for ax, metric in zip(axes, metrics_avail):
        sorted_signals = signals.sort_values(metric, ascending=False).reset_index(drop=True)
        target_rank = sorted_signals[sorted_signals['signal_id'] == SCREENER_ID].index
        colors = ['#e74c3c' if s == SCREENER_ID else '#bdc3c7' for s in sorted_signals['signal_id']]
        ax.bar(sorted_signals['signal_id'], sorted_signals[metric], color=colors, edgecolor='none')
        ax.set_xticklabels(sorted_signals['signal_id'], rotation=45, ha='right', fontsize=7)
        ax.set_title(metric, fontsize=9)
        ax.axhline(0, color='gray', linewidth=0.8)

    plt.suptitle(f"'{SCREENER_ID}' vs All Screeners (red = target)", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── SHAP importance for screener's horizon (if ML screener) ───────────────────
if not screener_matches.empty:
    # Determine horizon from screener id (ml_3y_oof -> 3y; alpha_value -> use LIVE_HORIZON)
    horizon_map = {'ml_6m_oof': '6m', 'ml_1y_oof': '1y', 'ml_2y_oof': '2y',
                   'ml_3y_oof': '3y', 'ml_5y_oof': '5y'}
    scr_horizon = horizon_map.get(SCREENER_ID, LIVE_HORIZON)

    shap_path = REPORTS / f'shap_importance_{scr_horizon}.csv'
    feat_path = REPORTS / f'feature_importance_{scr_horizon}.csv'

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    if shap_path.exists():
        df_shap = pd.read_csv(shap_path).head(TOP_FEATURES_N).sort_values('shap_mean_abs')
        axes[0].barh(df_shap['feature'], df_shap['shap_mean_abs'], color='steelblue')
        axes[0].set_title(f'SHAP Importance — {scr_horizon}')
        axes[0].set_xlabel('Mean |SHAP|')
        axes[0].tick_params(axis='y', labelsize=8)
    else:
        axes[0].text(0.5, 0.5, f'shap_importance_{scr_horizon}.csv not found',
                     ha='center', va='center', transform=axes[0].transAxes)

    if feat_path.exists():
        df_feat = pd.read_csv(feat_path).head(TOP_FEATURES_N).sort_values('importance')
        axes[1].barh(df_feat['feature'], df_feat['importance'], color='coral')
        axes[1].set_title(f'LightGBM Importance — {scr_horizon}')
        axes[1].set_xlabel('Importance (split count)')
        axes[1].tick_params(axis='y', labelsize=8)
    else:
        axes[1].text(0.5, 0.5, f'feature_importance_{scr_horizon}.csv not found',
                     ha='center', va='center', transform=axes[1].transAxes)

    plt.suptitle(f"Feature Importance for '{SCREENER_ID}' (horizon={scr_horizon})", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Walk-forward AUC for this screener's horizon ─────────────────────────────
if not screener_matches.empty:
    wf_path = REPORTS / f'walk_forward_auc_{scr_horizon}.csv'
    oof_path = REPORTS / f'oof_auc_{scr_horizon}.csv'

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    if wf_path.exists():
        df_wf = pd.read_csv(wf_path).sort_values('test_year')
        axes[0].plot(df_wf['test_year'], df_wf['auc'], marker='o', color='steelblue')
        axes[0].axhline(0.50, color='gray', linestyle=':', linewidth=1)
        axes[0].axhline(df_wf['auc'].mean(), color='orange', linestyle='--',
                        label=f'Mean={df_wf["auc"].mean():.3f}')
        axes[0].set_xlabel('Test Year')
        axes[0].set_ylabel('AUC')
        axes[0].set_title(f'Walk-Forward AUC — {scr_horizon}')
        axes[0].legend()
        axes[0].set_ylim(0.45, 0.75)

    if oof_path.exists():
        df_oof = pd.read_csv(oof_path).sort_values('test_year')
        axes[1].plot(df_oof['test_year'], df_oof['auc'], marker='s', color='coral')
        axes[1].axhline(0.50, color='gray', linestyle=':', linewidth=1)
        axes[1].axhline(df_oof['auc'].mean(), color='orange', linestyle='--',
                        label=f'Mean={df_oof["auc"].mean():.3f}')
        axes[1].set_xlabel('Test Year')
        axes[1].set_ylabel('AUC')
        axes[1].set_title(f'OOF AUC — {scr_horizon}')
        axes[1].legend()
        axes[1].set_ylim(0.45, 0.75)

    plt.suptitle(f"Model Stability: '{SCREENER_ID}'", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Screener bootstrap confidence summary ────────────────────────────────────
if not screener_matches.empty:
    print(f"\n{'─'*50}")
    print(f"Bootstrap confidence for '{SCREENER_ID}'")
    print(f"{'─'*50}")

    bs_fields = [
        ('CAGR point estimate',     'cagr_pct',                   '%'),
        ('CAGR bootstrap mean',     'cagr_bootstrap_mean_pct',    '%'),
        ('CAGR ± 1σ',              'cagr_bootstrap_1sigma_pct',   '%'),
        ('Sharpe point estimate',   'sharpe',                     ''),
        ('Sharpe bootstrap mean',   'sharpe_bootstrap_mean',      ''),
        ('Sharpe ± 1σ',            'sharpe_bootstrap_1sigma',     ''),
        ('Max drawdown',           'max_drawdown_pct',             '%'),
        ('Excess CAGR vs SPY',     'excess_cagr_vs_spy',          '%'),
        ('Hit rate',               'hit_rate_pct',                 '%'),
        ('Beta vs SPY',            'beta_vs_spy',                  ''),
    ]
    for label, field, unit in bs_fields:
        val = scr.get(field, None)
        if val is not None and pd.notna(val):
            print(f"  {label:35} {val:>8.3f}{unit}")

---
## 5. Live Picks — Today's Candidates
> **Run after the weekly data refresh.** Uses quality gates + composite alpha score.

In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────────
if USE_HF:
    from huggingface_hub import hf_hub_download
    local_path = hf_hub_download(repo_id=HF_REPO, filename='historical_dataset_clean.parquet',
                                 repo_type='dataset')
    df_full = pd.read_parquet(local_path)
else:
    df_full = pd.read_parquet(DATA / 'historical_dataset_clean.parquet')

print(f'Dataset loaded: {df_full.shape}')
print(f'Markets: {df_full["market"].value_counts().to_dict()}')

In [ ]:
# ── Get latest annual row per ticker ─────────────────────────────────────────
df = df_full[df_full['period_type'] == 'annual'].copy()
df = (
    df.sort_values('fiscal_year', ascending=False)
      .groupby(['ticker', 'market'], sort=False)
      .first()
      .reset_index()
)
if MARKETS:
    df = df[df['market'].isin(MARKETS)]
print(f'Latest annual rows: {len(df):,}')

# ── Size filter ───────────────────────────────────────────────────────────────
n0 = len(df)
if 'market_cap_at_filing' in df.columns:
    df = df[df['market_cap_at_filing'].notna() & (df['market_cap_at_filing'] >= MIN_MARKET_CAP)]
    if MAX_MARKET_CAP:
        df = df[df['market_cap_at_filing'] <= MAX_MARKET_CAP]
print(f'After size filter: {len(df):,} / {n0:,}')

# ── Quality gates ─────────────────────────────────────────────────────────────
n1 = len(df)
if 'piotroski_f_score' in df.columns: df = df[df['piotroski_f_score'] >= PIOTROSKI_MIN]
if 'beneish_m_score'   in df.columns: df = df[df['beneish_m_score']   <  BENEISH_MAX]
if 'altman_z_score'    in df.columns: df = df[df['altman_z_score']    >  ALTMAN_MIN]
print(f'After quality gates: {len(df):,} / {n1:,}')

# ── Margin of safety ─────────────────────────────────────────────────────────
n2 = len(df)
if APPLY_MOS:
    mos = pd.Series(True, index=df.index)
    if 'price_to_book' in df.columns: mos &= df['price_to_book'].isna() | (df['price_to_book'] <= PB_MAX)
    if 'fcf_yield'     in df.columns: mos &= df['fcf_yield'].isna()      | (df['fcf_yield']     >= FCF_YIELD_MIN)
    df = df[mos]
    print(f'After MoS gates: {len(df):,} / {n2:,}')

print(f'\nScreened universe: {len(df):,} stocks')

In [ ]:
# ── IC-weighted composite alpha ───────────────────────────────────────────────
# Pull IC weights from registry for selected signals
ic_weights = {
    row['signal_id']: row['ic_mean']
    for _, row in signals[signals['selected']].iterrows()
    if pd.notna(row.get('ic_mean'))
}
print(f'IC weights used: {ic_weights}')

def compute_composite(df, weights):
    composite = pd.Series(0.0, index=df.index)
    weight_sum = pd.Series(0.0, index=df.index)
    for col, w in weights.items():
        if col in df.columns:
            ranks = df[col].rank(pct=True)
            valid = ranks.notna()
            composite[valid] += ranks[valid] * w
            weight_sum[valid] += w
    return composite / weight_sum.clip(lower=1e-9)

df['composite_score'] = compute_composite(df, ic_weights)
df['label'] = df['ticker'] + ' (' + df['market'] + ')'
df['leverage_candidate'] = (
    df['beta_12m'].notna() & (df['beta_12m'] < BETA_MAX)
) if 'beta_12m' in df.columns else False

# Kelly sizing
f_full = (2.0 * df['composite_score'] - 1.0).clip(lower=0.0)
f_kelly = (f_full * HALF_KELLY).clip(upper=POSITION_CAP)
total = f_kelly.sum()
df['kelly_pct'] = ((f_kelly / total * 100) if total > 0 else f_kelly).round(2)

print(f'\nComposite score: mean={df["composite_score"].mean():.3f}  p90={df["composite_score"].quantile(0.9):.3f}')

In [ ]:
# ── Top LONG picks ────────────────────────────────────────────────────────────
display_cols = [
    'label', 'fiscal_year', 'market', 'composite_score',
    'alpha_value', 'alpha_quality', 'alpha_fraud_risk', 'ml_3y_oof',
    'piotroski_f_score', 'beneish_m_score', 'altman_z_score',
    'price_to_book', 'fcf_yield', 'market_cap_at_filing',
    'beta_12m', 'leverage_candidate', 'kelly_pct',
]
present_dc = [c for c in display_cols if c in df.columns]

longs = df.nlargest(TOP_N_LONG, 'composite_score')[present_dc].reset_index(drop=True)
if 'market_cap_at_filing' in longs.columns:
    longs['mktcap_$M'] = (longs['market_cap_at_filing'] / 1e6).round(1)
    longs = longs.drop(columns=['market_cap_at_filing'])
longs.index = range(1, len(longs) + 1)

print(f'Top {TOP_N_LONG} LONG candidates:')
longs

In [ ]:
# ── Top SHORT picks ───────────────────────────────────────────────────────────
short_cols = [
    'label', 'fiscal_year', 'market', 'composite_score',
    'alpha_fraud_risk', 'ml_3y_oof',
    'piotroski_f_score', 'beneish_m_score', 'altman_z_score',
    'fraud_suspect', 'montier_c_score', 'price_to_book', 'market_cap_at_filing',
]
short_present = [c for c in short_cols if c in df.columns]
shorts = df.nsmallest(TOP_N_SHORT, 'composite_score')[short_present].reset_index(drop=True)
if 'market_cap_at_filing' in shorts.columns:
    shorts['mktcap_$M'] = (shorts['market_cap_at_filing'] / 1e6).round(1)
    shorts = shorts.drop(columns=['market_cap_at_filing'])
shorts.index = range(1, len(shorts) + 1)

print(f'Top {TOP_N_SHORT} SHORT candidates:')
shorts

In [ ]:
# ── Portfolio summary + charts ────────────────────────────────────────────────
top_longs_df = df.nlargest(TOP_N_LONG, 'composite_score')

print(f"{'═'*50}")
print('PORTFOLIO SUMMARY')
print(f"{'═'*50}")
print(f'  Universe (screened) : {len(df):,} stocks')
print(f'  Long picks          : {TOP_N_LONG}')
print(f'  Short picks         : {TOP_N_SHORT}')
print(f'  Horizon             : {LIVE_HORIZON}')
print()
print('Market breakdown (longs):')
print(top_longs_df['market'].value_counts().to_string())

if 'leverage_candidate' in top_longs_df.columns:
    print(f"\nLeverage-eligible: {top_longs_df['leverage_candidate'].sum()} / {TOP_N_LONG}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['composite_score'], bins=40, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(top_longs_df['composite_score'].min(), color='green', linestyle='--', label='Long cutoff')
axes[0].set_title('Composite Score Distribution')
axes[0].legend()

mc = top_longs_df['market'].value_counts()
axes[1].bar(mc.index, mc.values, color='steelblue', edgecolor='black')
axes[1].set_title(f'Long Picks by Market (top {TOP_N_LONG})')

kw = top_longs_df.nlargest(TOP_N_LONG, 'composite_score')['kelly_pct'].values
lev_flags = top_longs_df.nlargest(TOP_N_LONG, 'composite_score')['leverage_candidate'].values \
    if 'leverage_candidate' in top_longs_df.columns else [False] * TOP_N_LONG
bar_colors = ['orange' if lc else 'steelblue' for lc in lev_flags]
axes[2].bar(range(len(kw)), kw, color=bar_colors)
axes[2].set_title('Kelly Weights — Long Picks\n(orange = leverage candidate)')
axes[2].set_xlabel('Rank')

plt.tight_layout()
plt.show()

In [ ]:
# ── Export picks ──────────────────────────────────────────────────────────────
from datetime import date
today = date.today().isoformat()

longs.to_csv(ROOT / 'reports' / f'screener_longs_{today}.csv', index=False)
shorts.to_csv(ROOT / 'reports' / f'screener_shorts_{today}.csv', index=False)
print(f'Exported to reports/screener_longs_{today}.csv and screener_shorts_{today}.csv')